In [ ]:
import requests
import pandas as pd
import time
from io import BytesIO
from datetime import datetime

import sys
sys.path.append('..')
from utils.bucket_utils import get_duck_con

In [1]:
print("hello")

hello


In [7]:
# ==========================================
# 📍 2. ตั้งค่าพิกัดจังหวัดปทุมธานี (7 อำเภอ)
# ==========================================
locations = [
    {"name": "เมืองปทุมธานี", "lat": 14.0208, "lon": 100.5250},
    {"name": "คลองหลวง", "lat": 14.0645, "lon": 100.6436},
    {"name": "ธัญบุรี", "lat": 14.0206, "lon": 100.7350},
    {"name": "หนองเสือ", "lat": 14.1618, "lon": 100.8252},
    {"name": "ลาดหลุมแก้ว", "lat": 14.0381, "lon": 100.4143},
    {"name": "ลำลูกกา", "lat": 13.9312, "lon": 100.7513},
    {"name": "สามโคก", "lat": 14.0642, "lon": 100.5240}
]

lats = ",".join([str(loc["lat"]) for loc in locations])
lons = ",".join([str(loc["lon"]) for loc in locations])

# ==========================================
# 🔄 3. ฟังก์ชันหลักสำหรับดึงข้อมูลและอัปโหลด
# ==========================================
def fetch_and_upload_to_minio():
    now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f"[{now_str}] ⏳ กำลังดึงข้อมูลสภาพอากาศและ PM2.5...")

    # URL สำหรับ 2 APIs
    weather_url = "https://api.open-meteo.com/v1/forecast"
    aq_url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    
    # พารามิเตอร์สภาพอากาศ (เหมือนเดิม)
    weather_params = {
        "latitude": lats,
        "longitude": lons,
        "current": "temperature_2m,relative_humidity_2m,apparent_temperature,is_day,precipitation,rain,weather_code,cloud_cover,pressure_msl,surface_pressure,wind_speed_10m,wind_direction_10m,wind_gusts_10m",
        "timezone": "Asia/Bangkok"
    }

    # พารามิเตอร์คุณภาพอากาศ (เพิ่ม pm2_5 และ pm10 เข้าไปเผื่อไว้ครับ)
    aq_params = {
        "latitude": lats,
        "longitude": lons,
        "current": "pm2_5,pm10",
        "timezone": "Asia/Bangkok"
    }

    try:
        # ยิง API ทั้ง 2 เส้น
        weather_res = requests.get(weather_url, params=weather_params)
        aq_res = requests.get(aq_url, params=aq_params)
        
        weather_res.raise_for_status()
        aq_res.raise_for_status()
        
        weather_data = weather_res.json()
        aq_data = aq_res.json()
        
        all_combined_data = []
        
        # ตรวจสอบว่าดึงมาหลายพิกัด (เป็น List)
        if isinstance(weather_data, list) and isinstance(aq_data, list):
            # ใช้ zip เพื่อดึงข้อมูลของแต่ละอำเภอจาก 2 API มาพร้อมๆ กัน
            for i, (w_dist, aq_dist) in enumerate(zip(weather_data, aq_data)):
                district_name = locations[i]["name"]
                w_current = w_dist["current"]
                aq_current = aq_dist["current"]
                
                # นำข้อมูลสภาพอากาศและคุณภาพอากาศมารวมกันใน Record เดียว
                record = {
                    "เวลา (อัปเดต)": w_current["time"],
                    "อำเภอ": district_name,
                    "อุณหภูมิ (°C)": w_current["temperature_2m"],
                    "อุณหภูมิที่รู้สึกได้ (°C)": w_current["apparent_temperature"],
                    "ความชื้น (%)": w_current["relative_humidity_2m"],
                    "ปริมาณฝน (mm)": w_current["precipitation"],
                    "รหัสสภาพอากาศ": w_current["weather_code"],
                    "เมฆปกคลุม (%)": w_current["cloud_cover"],
                    "ความเร็วลม (km/h)": w_current["wind_speed_10m"],
                    "PM2.5 (μg/m³)": aq_current["pm2_5"], # ดึงค่า PM2.5 มาใส่
                    "PM10 (μg/m³)": aq_current["pm10"]    # ดึงค่า PM10 มาใส่
                }
                all_combined_data.append(record)

        # แปลงเป็น DataFrame และเซฟขึ้น MinIO เหมือนเดิม
        df = pd.DataFrame(all_combined_data)
        
        csv_buffer = BytesIO()
        df.to_csv(csv_buffer, index=False, encoding='utf-8-sig')
        csv_buffer.seek(0)

        file_name = f"pathum_weather_aqi_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"        
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ อัปโหลดไฟล์ '{file_name}' (พร้อมข้อมูลฝุ่น) ลง MinIO สำเร็จ!\n")
        return df
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาด: {e}")

In [8]:
df = fetch_and_upload_to_minio()

[2026-03-26 15:38:41] ⏳ กำลังดึงข้อมูลสภาพอากาศและ PM2.5...
[15:38:43] ✅ อัปโหลดไฟล์ 'pathum_weather_aqi_20260326_153843.csv' (พร้อมข้อมูลฝุ่น) ลง MinIO สำเร็จ!



In [9]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   เวลา (อัปเดต)              7 non-null      object 
 1   อำเภอ                      7 non-null      object 
 2   อุณหภูมิ (°C)              7 non-null      float64
 3   อุณหภูมิที่รู้สึกได้ (°C)  7 non-null      float64
 4   ความชื้น (%)               7 non-null      int64  
 5   ปริมาณฝน (mm)              7 non-null      float64
 6   รหัสสภาพอากาศ              7 non-null      int64  
 7   เมฆปกคลุม (%)              7 non-null      int64  
 8   ความเร็วลม (km/h)          7 non-null      float64
 9   PM2.5 (μg/m³)              7 non-null      float64
 10  PM10 (μg/m³)               7 non-null      float64
dtypes: float64(6), int64(3), object(2)
memory usage: 744.0+ bytes


,เวลา (อัปเดต),อำเภอ,อุณหภูมิ (°C),อุณหภูมิที่รู้สึกได้ (°C),ความชื้น (%),ปริมาณฝน (mm),รหัสสภาพอากาศ,เมฆปกคลุม (%),ความเร็วลม (km/h),PM2.5 (μg/m³),PM10 (μg/m³)
0,2026-03-26T15:30,เมืองปทุมธานี,36.9,40.0,39,0.0,2,67,8.6,31.8,34.1
1,2026-03-26T15:30,คลองหลวง,37.5,41.0,37,0.0,2,58,5.8,27.9,30.3
2,2026-03-26T15:30,ธัญบุรี,37.0,40.6,40,0.0,1,38,9.4,27.9,30.3
3,2026-03-26T15:30,หนองเสือ,37.3,40.7,38,0.0,1,42,7.2,27.9,30.3
4,2026-03-26T15:30,ลาดหลุมแก้ว,37.1,40.0,38,0.0,2,55,8.0,31.8,34.1


In [11]:
# Connect with Google Bucket through DuckDB
con = get_duck_con()

In [ ]:
# Upload data (df) to Google Bucket
path = 's3://pea-oms/test.parquet'
con.execute(f"""
    COPY (
        SELECT *
        FROM df
    )
    TO '{path}' (FORMAT parquet)
""")

In [14]:
# Test read data from Google Bucket
con.execute(f"""
    SELECT * 
    FROM read_parquet('{path}')       
""").df()

,เวลา (อัปเดต),อำเภอ,อุณหภูมิ (°C),อุณหภูมิที่รู้สึกได้ (°C),ความชื้น (%),ปริมาณฝน (mm),รหัสสภาพอากาศ,เมฆปกคลุม (%),ความเร็วลม (km/h),PM2.5 (μg/m³),PM10 (μg/m³)
0,2026-03-26T15:30,เมืองปทุมธานี,36.9,40.0,39,0.0,2,67,8.6,31.8,34.1
1,2026-03-26T15:30,คลองหลวง,37.5,41.0,37,0.0,2,58,5.8,27.9,30.3
2,2026-03-26T15:30,ธัญบุรี,37.0,40.6,40,0.0,1,38,9.4,27.9,30.3
3,2026-03-26T15:30,หนองเสือ,37.3,40.7,38,0.0,1,42,7.2,27.9,30.3
4,2026-03-26T15:30,ลาดหลุมแก้ว,37.1,40.0,38,0.0,2,55,8.0,31.8,34.1
5,2026-03-26T15:30,ลำลูกกา,35.7,39.7,46,0.0,2,53,10.9,27.9,30.3
6,2026-03-26T15:30,สามโคก,37.2,40.4,38,0.0,1,41,6.6,31.8,34.1
